In [7]:
import pandas as pd
import re
import os
from pathlib import Path

# Diretório base e utilitário para dados
BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'Banco_de_Dados_PII3_AWS').exists() and (BASE_DIR.parent / 'Banco_de_Dados_PII3_AWS').exists():
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'Banco_de_Dados_PII3_AWS'
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Diretório de dados não encontrado: {DATA_DIR}")

def get_data_path(nome_arquivo):
    return DATA_DIR / nome_arquivo

In [ ]:
def limpar_avaliacoes(texto):
    if not isinstance(texto, str):
        return texto

    texto = texto.strip()
    texto = re.sub(r'^[\W_]+', '', texto, flags=re.UNICODE)
    texto = re.sub(r'\s{2,}', ' ', texto)

    # Remove espacos antes de pontuacao e reduz repeticoes
    texto = re.sub(r'\s+([.,;:!?])', r'\1', texto)
    texto = re.sub(r'([.,;:!?])\1+', r'\1', texto)
    texto = re.sub(r'([.,;:!?])(?=[^\s])', r'\1 ', texto)

    # Padroniza abreviacoes comuns
    mapeamento = {
        r'\b(n[ãa]o|nã|nao)\b': 'não',
        r'\bvc\b': 'você',
        r'\bvcs\b': 'vocês',
        r'\b(mt|mto)\b': 'muito'
    }

    def aplicar_mapeamento(match):
        original = match.group(0)
        alvo = ''
        for padrao, subst in mapeamento.items():
            if re.search(padrao, original, flags=re.IGNORECASE):
                alvo = subst
                break

        if original.isupper():
            return alvo.upper()
        if original and original[0].isupper():
            return alvo.capitalize()
        return alvo

    regex_completo = '|'.join(mapeamento.keys())
    texto = re.sub(regex_completo, aplicar_mapeamento, texto, flags=re.IGNORECASE)

    texto = re.sub(r'(\.\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), texto)
    texto = re.sub(r'[\s]+$', '', texto)
    return texto



In [3]:
def juntar_titulo_mensagem(df, coluna_titulo='review_comment_title', coluna_mensagem='review_comment_message'):
    if coluna_titulo not in df.columns or coluna_mensagem not in df.columns:
        print(f'Erro: colunas {coluna_titulo} ou {coluna_mensagem} nao encontradas.')
        return df

    df[coluna_titulo] = df[coluna_titulo].astype('string')
    df[coluna_mensagem] = df[coluna_mensagem].astype('string')

    def combinar(titulo, mensagem):
        titulo = str(titulo).strip() if pd.notna(titulo) else ''
        mensagem = str(mensagem).strip() if pd.notna(mensagem) else ''

        if titulo and mensagem:
            return f'{titulo} - {mensagem}'
        return titulo or mensagem

    df[coluna_mensagem] = df.apply(lambda row: combinar(row[coluna_titulo], row[coluna_mensagem]), axis=1)
    return df

In [4]:
def processar_csv(caminho_arquivo, nome_coluna):
    # aceita string ou Path
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo nao encontrado em {caminho}')
        return

    df = pd.read_csv(caminho)

    if nome_coluna in df.columns:
        print(f'Limpando espacos e padronizando a coluna {nome_coluna}...')
        df[nome_coluna] = df[nome_coluna].apply(limpar_avaliacoes)
        df.to_csv(caminho, index=False)
        print('Sucesso! Espacos duplos removidos e arquivo atualizado.')
    else:
        print(f'Erro: A coluna {nome_coluna} nao existe no arquivo CSV.')

In [9]:
def remover_linhas_sem_review(caminho_arquivo, coluna='review_comment_message'):
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo nao encontrado em {caminho}')
        return

    df = pd.read_csv(caminho)
    if coluna not in df.columns:
        print(f'Erro: A coluna {coluna} nao existe no arquivo CSV.')
        return

    antes = len(df)
    df[coluna] = df[coluna].astype('string')
    df = df[df[coluna].str.strip() != '']
    depois = len(df)
    removidas = antes - depois

    df.to_csv(caminho, index=False)
    print(f'Removidas {removidas} linhas sem valor em {coluna}.')
    return df

In [ ]:
# Exemplo de uso:
df = pd.read_csv(get_data_path('avaliacoes.csv'))
df['review_comment_title'] = df['review_comment_title'].apply(limpar_avaliacoes)
df['review_comment_message'] = df['review_comment_message'].apply(limpar_avaliacoes)
df = juntar_titulo_mensagem(df)
df.to_csv(get_data_path('avaliacoes.csv'), index=False)

In [10]:
remover_linhas_sem_review(get_data_path('avaliacoes.csv'))

Removidas 56656 linhas sem valor em review_comment_message.


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,recomendo - aparelho eficiente. No site a marc...,2018-05-22 00:00:00,2018-05-23 16:45:47
12,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,NaN,"Mas um pouco, travando. Pelo valor ta Boa.",2018-02-16 00:00:00,2018-02-20 10:52:22
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Super recomendo - Vendedor confiável, produto ...",2018-05-23 00:00:00,2018-05-24 03:00:01
...,...,...,...,...,...,...,...
99205,98fffa80dc9acbde7388bef1600f3b15,d398e9c82363c12527f71801bf0e6100,4,NaN,para este produto recebi de acordo com a compr...,2017-11-29 00:00:00,2017-11-30 15:52:51
99208,df5fae90e85354241d5d64a8955b2b09,509b86c65fe4e2ad5b96408cfef9755e,5,NaN,Entregou dentro do prazo. O produto chegou em ...,2018-02-07 00:00:00,2018-02-19 19:47:23
99215,a709d176f59bc3af77f4149c96bae357,d5cb12269711bd1eaf7eed8fd32a7c95,3,NaN,"O produto não foi enviado com NF, não existe v...",2018-05-19 00:00:00,2018-05-20 21:51:06
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43


In [ ]:
def converter_para_string(df, nome_coluna):
    """
    Converte uma coluna do tipo object para string.
    """
    if nome_coluna in df.columns:
        df[nome_coluna] = df[nome_coluna].astype("string")
        print(f"Coluna '{nome_coluna}' convertida para string.")
    else:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
    return df

df = pd.read_csv(get_data_path('avaliacoes.csv'))

converter_para_string(df, 'order_id')

In [ ]:
def converter_para_datetime(df, nome_coluna):
    """
    Converte uma coluna do tipo object para datetime.
    """
    if nome_coluna in df.columns:
        # errors='coerce' transforma valores inválidos em nulos (NaT)
        df[nome_coluna] = pd.to_datetime(df[nome_coluna], errors='coerce')
        print(f"Coluna '{nome_coluna}' convertida para datetime.")
    else:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
    return df

converter_para_datetime(df, 'review_answer_timestamp')